In [73]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import ttest_ind
import pingouin as pg
from statsmodels.multivariate.manova import MANOVA

In [ ]:
df = pd.read_csv("")
df.columns.values[13] = 'group_number'

In [ ]:
mapping = {
    1: ('left', 'low', 'human'),
    2: ('left', 'low', 'LLM'),
    3: ('left', 'low', 'control'),
    4: ('left', 'medium', 'human'),
    5: ('left', 'medium', 'LLM'),
    6: ('left', 'medium', 'control'),
    7: ('left', 'high', 'human'),
    8: ('left', 'high', 'LLM'),
    9: ('left', 'high', 'control'),

    10: ('center', 'low', 'human'),
    11: ('center', 'low', 'LLM'),
    12: ('center', 'low', 'control'),
    13: ('center', 'medium', 'human'),
    14: ('center', 'medium', 'LLM'),
    15: ('center', 'medium', 'control'),
    16: ('center', 'high', 'human'),
    17: ('center', 'high', 'LLM'),
    18: ('center', 'high', 'control'),

    19: ('right', 'low', 'human'),
    20: ('right', 'low', 'LLM'),
    21: ('right', 'low', 'control'),
    22: ('right', 'medium', 'human'),
    23: ('right', 'medium', 'LLM'),
    24: ('right', 'medium', 'control'),
    25: ('right', 'high', 'human'),
    26: ('right', 'high', 'LLM'),
    27: ('right', 'high', 'control'),
}

# Apply the mapping to create new columns
df[['article_orientation', 'article_bias_level', 'explanation_type']] = df['group_number'].map(mapping).apply(pd.Series)

In [ ]:
df = df.rename(columns={
    "After reading the explanation, please tell us what you think about the following sentence: In my opinion, this article is biased.": "bias_t2",
    "Please tell us what you think about the following sentence: In my opinion, this article is biased.": "bias_t1",
    "Do you consider yourself to be liberal, conservative or somewhere in between?   [Political Orientation|Liberal|Conservative]": "Participant_Political_Lean",
    "The explanation was useful. [agree|disagree]": "Usefulness",
    "The explanation was complete. [agree|disagree]": "Completeness"
})

In [ ]:
# UEQ Score
# Step 1: Reverse-code the negatively-keyed items
df["valuable_rc"] = -1 * df["The explanation was… [valuable|inferior]"]
df["good_rc"] = -1 * df["The explanation was… [good|bad]"]
df["clear_rc"] = -1 * df["The explanation was… [clear|confusing]"]

# Step 2: List of correctly-coded items (original or reversed)
ueq_items = [
    "The explanation was… [not understandable|understandable]",
    "valuable_rc",
    "The explanation was… [obstructive|supportive]",
    "good_rc",
    "The explanation was… [complicated|easy]",
    "clear_rc"
]

# Step 3: Compute the mean UEQ score per participant
df["UEQ"] = df[ueq_items].mean(axis=1)


In [ ]:
relevant_columns = [
    'article_orientation', 
    'article_bias_level',             
    'explanation_type',    
    'bias_t1',
    'bias_t2',
    'Participant_Political_Lean',
    'Usefulness',
    'Completeness', 
    'UEQ'
]

df_relevant = df[relevant_columns].copy()

In [59]:
import random

orientations = ['left', 'center', 'right']
bias_levels = ['low', 'medium', 'high']
explanation_types = ['human', 'LLM', 'control']
bias_scale = ['Strongly disagree', 'Disagree', 'Somewhat disagree', 'Somewhat agree', 'Agree', 'Strongly agree']
lean_range = list(range(-10, 11))
score_range = list(range(-3, 4))

# Mapping from earlier
mapping = {
    i + 1: (o, b, e)
    for i, (o, b, e) in enumerate(
        (o, b, e)
        for o in orientations
        for b in bias_levels
        for e in explanation_types
    )
}

# Generate dummy data
dummy_data = []
for i in range(1, 28):
    orientation, bias_level, explanation = mapping[i]
    row = {
        "article_orientation": orientation,
        "article_bias_level": bias_level,
        "explanation_type": explanation,
        "bias_t1": random.choice(bias_scale),
        "bias_t2": random.choice(bias_scale),
        "Participant_Political_Lean": random.choice(lean_range),
        "Usefulness": random.choice(score_range),
        "Completeness": random.choice(score_range)
    }
    dummy_data.append(row)

# Display as a table (pandas optional)
import pandas as pd
df_test = pd.DataFrame(dummy_data)
df_test['UEQ'] = np.random.randint(-3, 4, size=len(df_test))
print(df_test)



   article_orientation article_bias_level explanation_type            bias_t1  \
0                 left                low            human  Somewhat disagree   
1                 left                low              LLM              Agree   
2                 left                low          control              Agree   
3                 left             medium            human  Strongly disagree   
4                 left             medium              LLM     Somewhat agree   
5                 left             medium          control  Strongly disagree   
6                 left               high            human  Somewhat disagree   
7                 left               high              LLM              Agree   
8                 left               high          control              Agree   
9               center                low            human  Strongly disagree   
10              center                low              LLM  Strongly disagree   
11              center      

In [60]:
df_relevant = df_test.copy()

In [61]:
bias_scale = {
    "Strongly disagree": -3,
    "Disagree": -2,
    "Somewhat disagree": -1,
    "Somewhat agree": 1,
    "Agree": 2,
    "Strongly agree": 3
}

# Map responses in bias_t1 and bias_t2 to numeric values
df_relevant['bias_t1_num'] = df_relevant['bias_t1'].map(bias_scale)
df_relevant['bias_t2_num'] = df_relevant['bias_t2'].map(bias_scale)

# Calculate the change in perceived bias
df_relevant['BiasChange'] = df_relevant['bias_t2_num'] - df_relevant['bias_t1_num']

In [62]:
df_relevant.head()

,article_orientation,article_bias_level,explanation_type,bias_t1,bias_t2,Participant_Political_Lean,Usefulness,Completeness,bias_t1_num,bias_t2_num,BiasChange
0,left,low,human,Somewhat disagree,Somewhat disagree,10,2,1,-1,-1,0
1,left,low,LLM,Agree,Agree,8,-1,-1,2,2,0
2,left,low,control,Agree,Strongly agree,-5,0,-3,2,3,1
3,left,medium,human,Strongly disagree,Somewhat agree,7,2,1,-3,1,4
4,left,medium,LLM,Somewhat agree,Disagree,3,-2,-1,1,-2,-3


In [63]:
df['explanation_type'] = df['explanation_type'].astype('category')
df['article_bias_level'] = df['article_bias_level'].astype('category')

In [65]:
#H1: Two-Way ANOVA on BiasChange and H2 Subset (Human vs. LLM Only) Two-Way ANOVA
model_h1 = smf.ols('BiasChange ~ C(explanation_type) * C(article_bias_level)', data=df_relevant).fit()
anova_h1 = sm.stats.anova_lm(model_h1, typ=2)
print(anova_h1)

# Post-hoc comparisons if significant
tukey_exp = pairwise_tukeyhsd(df_relevant['BiasChange'], df_relevant['explanation_type'])
tukey_bias = pairwise_tukeyhsd(df_relevant['BiasChange'], df_relevant['article_bias_level'])

print(tukey_exp)
print(tukey_bias)

                                               sum_sq    df         F  \
C(explanation_type)                         18.962963   2.0  0.977099   
C(article_bias_level)                        1.185185   2.0  0.061069   
C(explanation_type):C(article_bias_level)   33.481481   4.0  0.862595   
Residual                                   174.666667  18.0       NaN   

                                             PR(>F)  
C(explanation_type)                        0.395498  
C(article_bias_level)                      0.940953  
C(explanation_type):C(article_bias_level)  0.505002  
Residual                                        NaN  
 Multiple Comparison of Means - Tukey HSD, FWER=0.05 
 group1  group2 meandiff p-adj   lower  upper  reject
-----------------------------------------------------
    LLM control      0.0    1.0 -3.4768 3.4768  False
    LLM   human   1.7778 0.4214  -1.699 5.2545  False
control   human   1.7778 0.4214  -1.699 5.2545  False
----------------------------------------

In [74]:
# H3: One-Way MANOVA (UEQ, Usefulness, Completeness)

manova = MANOVA.from_formula('UEQ + Usefulness + Completeness ~ explanation_type', data=df_relevant)
print(manova.mv_test())

# Follow-up ANOVAs
for dv in ['UEQ', 'Usefulness', 'Completeness']:
    model = smf.ols(f'{dv} ~ C(explanation_type)', data=df_relevant).fit()
    anova = sm.stats.anova_lm(model, typ=2)
    print(f"\nANOVA for {dv}")
    print(anova)

    # Tukey HSD
    tukey = pairwise_tukeyhsd(df_relevant[dv], df_relevant['explanation_type'])
    print(tukey)


                 Multivariate linear model
                                                            
------------------------------------------------------------
       Intercept        Value  Num DF  Den DF F Value Pr > F
------------------------------------------------------------
          Wilks' lambda 0.7237 3.0000 22.0000  2.7999 0.0638
         Pillai's trace 0.2763 3.0000 22.0000  2.7999 0.0638
 Hotelling-Lawley trace 0.3818 3.0000 22.0000  2.7999 0.0638
    Roy's greatest root 0.3818 3.0000 22.0000  2.7999 0.0638
------------------------------------------------------------
                                                            
------------------------------------------------------------
    explanation_type    Value  Num DF  Den DF F Value Pr > F
------------------------------------------------------------
          Wilks' lambda 0.8475 6.0000 44.0000  0.6325 0.7034
         Pillai's trace 0.1586 6.0000 46.0000  0.6605 0.6816
 Hotelling-Lawley trace 0.1727 6.0000 27.6

In [71]:
# H4: Moderated Regression with Congruence
# Helper: assign article orientation to numeric direction
orientation_map = {'left': -1, 'center': 0, 'right': 1}
df_relevant['orientation_numeric'] = df_relevant['article_orientation'].map(orientation_map)

# Define congruence: congruent if orientation × lean > 0 or both zero
df_relevant['congruent'] = np.sign(df_relevant['orientation_numeric'] * df_relevant['Participant_Political_Lean'])
df_relevant['congruence'] = np.where(df_relevant['congruent'] > 0, 'Congruent', 'Incongruent')
df_relevant['congruence'] = df_relevant['congruence'].astype('category')


In [72]:
# Bias at T1 ~ political congruence
model_h4a = smf.ols('BiasChange ~ C(congruence)', data=df_relevant).fit()
print(sm.stats.anova_lm(model_h4a, typ=2))

# BiasChange ~ explanation_type * congruence
model_h4b = smf.ols('BiasChange ~ C(explanation_type) * C(congruence)', data=df_relevant).fit()
print(sm.stats.anova_lm(model_h4b, typ=2))

# Human-specific interaction
df_human = df_relevant[df_relevant['explanation_type'] == 'human']
model_h4c = smf.ols('BiasChange ~ C(congruence)', data=df_human).fit()
print(sm.stats.anova_lm(model_h4c, typ=2))

# For each UEQ variable
for dv in ['UEQ', 'Usefulness', 'Completeness']:
    model = smf.ols(f'{dv} ~ C(explanation_type) * C(congruence)', data=df_relevant).fit()
    print(f"\n{dv} ~ explanation_type × congruence")
    print(sm.stats.anova_lm(model, typ=2))


                   sum_sq    df         F    PR(>F)
C(congruence)    5.351852   1.0  0.600133  0.445793
Residual       222.944444  25.0       NaN       NaN
                                       sum_sq    df         F    PR(>F)
C(explanation_type)                 23.202991   2.0  1.307921  0.291519
C(congruence)                        9.591880   1.0  1.081362  0.310218
C(explanation_type):C(congruence)   13.467643   2.0  0.759153  0.480483
Residual                           186.273810  21.0       NaN       NaN
                  sum_sq   df        F    PR(>F)
C(congruence)  15.365079  1.0  1.26749  0.297351
Residual       84.857143  7.0      NaN       NaN

UEQ ~ explanation_type × congruence
                                      sum_sq    df         F    PR(>F)
C(explanation_type)                 3.185897   2.0  0.706056  0.504933
C(congruence)                       6.463675   1.0  2.864949  0.105314
C(explanation_type):C(congruence)  16.379976   2.0  3.630117  0.044254
Residual        

In [79]:
#H5 Bias at T1 matches Expert Asessment
"""
We should discuss this, not quite straightforward to align both scales. 
"""
bias_map = {'low': 1, 'medium': 3.5, 'high': 6}  # or your expert scale
df_relevant['ExpertBiasScore'] = df_relevant['article_bias_level'].map(bias_map)

# Optional: convert bias_t1 to a comparable numeric scale
bias_numeric_map = {
    "Strongly disagree": 1,
    "Disagree": 2,
    "Somewhat disagree": 3,
    "Somewhat agree": 4,
    "Agree": 5,
    "Strongly agree": 6
}
df_relevant['BiasT1Numeric'] = df_relevant['bias_t1'].map(bias_numeric_map)

# Independent t-test
t_stat, p_val = ttest_ind(df_relevant['BiasT1Numeric'], df_relevant['ExpertBiasScore'])
print(f"\nT-test BiasT1 vs ExpertBiasScore:\nt = {t_stat:.3f}, p = {p_val:.4f}")



T-test BiasT1 vs ExpertBiasScore:
t = 0.331, p = 0.7423
